video source: Q learning tutorial by Edan Meyer, using a block-pixel based videogame 
https://www.youtube.com/playlist?list=PL_49VD9KwQ_OpleGtJhWD24JDrrmXYXEQ

Code: https://github.com/ejmejm/Q-Learning-Tutorials/blob/master/DQN.ipynb

In [11]:
#import libraries

import torch
from torch import nn
import gymnasium as gym
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [8]:
#make first environments from gym catalogue -> picking the breakout paddle game option 

#env = gymnasium.make("BreakoutDeterministic-v4") ##error when running it because out outdated version, but still following for code references

## State space
## action space- what are the actions

In [10]:
#can read the action and observation space options from the environment 
#print(env.action_space)
#print(env.observation_space)

In [ ]:
## Get first observation 
obs = env.reset() ## started a new roll-out/epsidoe/playout of the game 
#print(obs) will show the current environemtn 

In [ ]:
### playout rounds of the game 

In [ ]:
#define max number of steps so game will acutally end
max_step= 1000
#env.reset()

for step in range(max_steps):
    #choose actions randomly 
    act = random.randint(0, 3) #0-3 inclusive

    #enact action 
    obs, reward, done, _= end.step(act) 
    #passes back -> next state/reward; reward for our action; done- whether this was the last state, and then extra info 

    #see it visually if it plays out 
    env.render()
    #add a delay when replaying it 
    time.sleep(0.05)

    if done:
        break




### setup deep Q net architectures 

In [13]:
## create a new class 
N_FRAMES = 3 #because 3 RGB channels
#given a current action and a current state, what reward can we expect to be generated?
class DQN(nn.Module): #trying to predict q values for a given action
    def __init__(self, n_acts):
        #convolusional neural network arcitechure -- blackbox that takes an input and gives an output
        super(DQN, self).__init__()

        ### 4 layers 
        self.layer1 = nn.Sequential(
            nn.Conv2d(N_FRAMES, 
                      #number of filters
                      16, 
                      #
                      kernel_size=8, stride=4, padding=0),
            nn.ReLU())
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=0),
            nn.ReLU())
        self.layer3 = nn.Sequential(
            #linear layer -- flattening the image I think 
            nn.Linear(32 * 12 * 9, 256),
            nn.ReLU())
        self.layer4 = nn.Sequential(
            #linear to the number of actions we have 
            nn.Linear(256, n_acts))

    #set the forward code 
    def forward(self, obs): #pass in observation from the environment 
        q_values = self.layer1(obs)
        q_values = self.layer2(q_values)
        
        # 2015 model: (32, 8x8, 4), (64, 4x4, 2), (64, 3x3, 1), (512)
        q_values = q_values.view(-1, 32 * 12 * 9)
        q_values = self.layer3(q_values) #fully connected layer 
        q_values = self.layer4(q_values)
        
        return q_values

In [14]:
### to test out the model 
dqn = DQN(n_acts=4)
input_data = torch.tensor(np.zeros((1, 3, 84, 110))).float()
#dqn([np.zeros((110, 84, 3))]) #define the shape of the image-- same shape as the input
dqn(input_data)

tensor([[ 0.0516,  0.0379, -0.0507,  0.0560]], grad_fn=<AddmmBackward0>)

The results is a tensor with 4 values. These are our estimated q-values for the 4 actions we took. Currently they are completely random, so we will change the code for the future 

In [ ]:
#get an observation 
obs= env.reset()
obs.shape #will be the RGB pixel

#define max number of steps so game will acutally end
max_step= 1000
#env.reset()

for step in range(max_steps):
    #choose actions randomly 
    #To not choose randomly, we will add 
    #q_values = dqn(filter(obs))[0]
    #act = np.argmax(q_values)
    act = random.randint(0, 3) #0-3 inclusive

    #enact action 
    obs, reward, done, _= end.step(act) 
    #passes back -> next state/reward; reward for our action; done- whether this was the last state, and then extra info 

    #see it visually if it plays out 
    env.render()
    #add a delay when replaying it 
    time.sleep(0.05)

    if done:
        break

### Next step: experience replay 
* normlaize pixels/environment to 0-1 --> change the range of pixel values from 0-1

* frame by frame we do not have any concept of what happened in the past --> so agent can see change over time, and then decide the order of which things are happening

* FOr a pixel based game, you will stack pixel frames (and resize as needed, and convert to grayscale
* 
* 

In [ ]:
## Added code: filter observation
N_FRAMES = 4

def filter_obs(obs, resize_shape=(84, 110), crop_shape=None):
    assert(type(obs) == np.ndarray), "The observation must be a numpy array!"
    assert(len(obs.shape) == 3), "The observation must be a 3D array!"

    obs = cv2.resize(obs, resize_shape, interpolation=cv2.INTER_LINEAR)
    obs = cv2.cvtColor(obs, cv2.COLOR_BGR2GRAY) #convert color to grayscale
    obs = obs / 255. #converting observation to the 0-1 normalization number 
    if crop_shape:
        crop_x_margin = (resize_shape[1] - crop_shape[1]) // 2
        crop_y_margin = (resize_shape[0] - crop_shape[0]) // 2
        
        x_start, x_end = crop_x_margin, resize_shape[1] - crop_x_margin
        y_start, y_end = crop_y_margin, resize_shape[0] - crop_y_margin
        
        obs = obs[x_start:x_end, y_start:y_end]
    
    return obs

def get_stacked_obs(obs, prev_frames): #stack the overvations -> single preprocessed observation 
    if not prev_frames: #would be an array that is passign the previous frames --> get the 3 previous frames
        prev_frames = [obs] * (N_FRAMES - 1)
        
    prev_frames.append(obs) #now make the 4th frame by the current observation state 

    #turning our previous frames into one frame --> this makes it easy to see the direction of movement of the ball 
    stacked_frames = np.stack(prev_frames) #change it from a list to a numpy list
    prev_frames = prev_frames[-(N_FRAMES-1):] 
    
    return stacked_frames, prev_frames

def preprocess_obs(obs, prev_frames):
    filtered_obs = filter_obs(obs)
    stacked_obs, prev_frames = get_stacked_obs(filtered_obs, prev_frames)
    return stacked_obs, prev_frames

### FOrmatting the reward -> want the reward to be constatn and normalized, helps keep things stable (less fine-tuing for individual environment)
## simple negative and positive reward 

def format_reward(reward):
    if reward > 0:
        return 1
    elif reward < 0:
        return -1
    return 0

### Experience replay 

* what stores the transitions and actions that have happened, so anytime we can train from previous data
* "off-policy training method" -> as you update your model, you keep traiing on old data that was genreated by an older model
* Then we rangomly take certain episodes -> get a good breath of possible expereinces, instead of just the current thing
* Helps bring up sample efficency, and other methods we will want to have
* So we do this by making a replay class 

In [1]:
class ExperienceReplay():
    def __init__(self, capacity): #capacity = how many transitions can actually be stored 
        self.capacity = capacity
        self.data = [] #empty at first, but this is where we will store the data 
        
    def add_step(self, step_data): #take the data and add it to our data storage 
        self.data.append(step_data)
        if len(self.data) > self.capacity: #want to make sure we do not go over the capacity 
            self.data = self.data[-self.capacity:] #if we are over capacity, we just remove the oldest stored data until we are within capacity 
            
    def sample(self, n): #give it the number of samples, and will return that many samples of previous experience
        n = min(n, len(self.data)) #how mahy samples we want 
        indices = np.random.choice(range(len(self.data)), n, replace=False) 
        samples = np.asarray(self.data)[indices]

        #convert the data to a tensor format, so we dont have to do that later on 
        #every step -> store the state, action, reward, the next state, and record the terminal data (is this the end of the episode -- important for training)
        state_data = torch.tensor(np.stack(samples[:, 0])).float().cuda() #using the GPU or CPU format- .cuda
        act_data = torch.tensor(np.stack(samples[:, 1])).long().cuda() #type long not floats 
        reward_data = torch.tensor(np.stack(samples[:, 2])).float().cuda()
        next_state_data = torch.tensor(np.stack(samples[:, 3])).float().cuda()
        terminal_data = torch.tensor(np.stack(samples[:, 4])).float().cuda()
        
        return state_data, act_data, reward_data, next_state_data, terminal_data

In [ ]:
er = ExperienceReplay(er_capacity)

----

### Training loop: notes

In [ ]:
### define some constants 
n_episodes = 10000 #how many total games we play out 
max_steps = 1000 #max number of steps in a single game

#depends on RAM -> how many possible things to put in the experience replay (64 GB RAM)
er_capacity = 50000 # 1m in paper ##

#Number of actions -> if discrete 
n_acts = env.action_space.n # 0: no-op, 1: start game, 2: right, 3: left

#requency of thigs
train_batch_size = 32
learning_rate = 2.5e-4
print_freq = 100 #printing out the updates as it trains -> sets how many games they print out 
update_freq = 4 #how commonly updating network
frame_skip = 3 ## you do not need to look at every frame in the game, only really skip over a few
n_anneal_steps = 1e5 # Anneal over 1m steps in paper 

## policy = how we decide what actions to take 
## denoted by (pi)
## given a state -> take the highest Q values
## always just take the best q-value without exploring -> the estimate could be wrong 
## pi(state) -> action or distribution of actions (probability it will take each action)

## epsilon greedy policy -> random action, otherwise take the action with the highest q-value 
epsilon = lambda step: np.clip(1 - 0.9 * (step/n_anneal_steps), 0.1, 1) # Anneal over 1m steps in paper, 100k here
## Have a number of annealing steps ->
## start out at 1 = taking a random action and then over time anneal the epsilon down over time
## balance between random choice to get experience and then switched to best q-value options



In [ ]:
## take our experience replay and teh DQN model 
er = ExperienceReplay(er_capacity)
model = DQN(n_acts=env.action_space.n).cuda()

## keep track of things -> total reward we get (to see/graph at the end of how we do)
all_rewards = []

## know how many actual steps we have taken, to help look at metrics and stuff
global_step = 0

#iterate: for each game (episode) 
for episode in range(n_episodes):
    prev_frames = [] #start a fresh frame
    #get the new observation of the episode/reset environemtn 
    obs, prev_frames = preprocess_obs(env.reset(), prev_frames)

    ### throughout a single episode-> keep track of the episode reward, and what step we are on
    episode_reward = 0
    step = 0

    
    while step < max_steps: #runs through all the steps in the episodes

        ### Enact a step ###
        ## When enacting a step -> first we want to choose an action 

        ## First check on our policy -> are on random seelction state or on smart policy (based on epsilon)
        
        if np.random.rand() < epsilon(global_step):
            act = np.random.choice(range(n_acts))
        else: ## The alternative -> 
            #set out observation and put it into our model to get our q-values
            obs_tensor = torch.tensor([obs]).float().cuda()
            q_values = model(obs_tensor)[0]
            q_values = q_values.cpu().detach().numpy()

            #Then we decide which action has the highest q-value
            act = np.argmax(q_values)

        #because we are doing a frame skip,
        ## we want to keep track of cumulative reward in a for loop
        cumulative_reward = 0
        for _ in range(frame_skip): 
            #for those 3 frames enact the same action every time ->
            ## get our next observation 
            ## get our calculated reward
            ## and evaluate whether or not it was a terminal state
            next_obs, reward, done, _ = env.step(act)
            cumulative_reward += reward
            
            if done or step >= max_steps:
                break

        ## want to make sure we are adding reward correctly -> add the cumulative reward to the episode reward
        ### Episode reward is a METRIC for how our agent is perfoming 
        ### format_reward is used during training stages
        episode_reward += cumulative_reward
        ## reward used for actual training 
        reward = format_reward(cumulative_reward)

        ### now we need to put this through our experience replay

        #preprocess our observation
        next_obs, prev_frames = preprocess_obs(next_obs, prev_frames)
        #add our observation to our experienced replay
        ## 
        er.add_step([obs, ## the old observation
                     act, ##the action we took
                     reward, ### the reward we got
                     next_obs, ## the observation that it led us too
                     int(done)]) ##a '1' if this is a terminal state (will come into play during training)
        ## set our current observation to our next observation 
        #update/reset the loop
        obs = next_obs

        ###---- Now we need to train our image so it improves each step, instead of just choosing randomly 
        ### Train on a minibatch ###

        if global_step % update_freq == 0:
            obs_data, act_data, reward_data, next_obs_data, terminal_data = er.sample(train_batch_size)
            model.train_on_batch(optimizer, obs_data, act_data, reward_data, next_obs_data, terminal_data)

        ## add the incremental calculations from the while loop
        step += 1
        global_step += 1
        
        if done:
            break

    ## at the end of each episode, append our reward
    ## so we can track how we are doing 
    all_rewards.append(episode_reward)


    ### Have a print out message so we can keep track of process as we train without fully rendering 
    
    if episode % print_freq == 0:
        print('Episode #{} | Step #{} | Epsilon {:.2f} | Avg. Reward {:.2f}'.format(
            episode, global_step, epsilon(global_step), np.mean(all_rewards[-print_freq:])))

## Implement the model ("render")

In [ ]:
prev_frames = []
obs, prev_frames = preprocess_obs(env.reset(), prev_frames)

for step in range(max_steps):
    if np.random.rand() < 0.05:
        act = np.random.choice(range(n_acts))
    else:
        obs_tensor = torch.tensor([obs]).float().cuda()
        q_values = model(obs_tensor)[0]
        q_values = q_values.cpu().detach().numpy()
        act = np.argmax(q_values)

    for _ in range(frame_skip):
        next_obs, reward, done, _ = env.step(act)
        if done or step >= max_steps:
            break
            
        env.render()
        time.sleep(0.05)
        
    if done:
        break

    obs, prev_frames = preprocess_obs(next_obs, prev_frames)

### Evaluate model performance 

In [ ]:
smoothed_rewards = []
smooth_window = 50
for i in range(smooth_window, len(all_rewards)-smooth_window):
    smoothed_rewards.append(np.mean(all_rewards[i-smooth_window:i+smooth_window]))
    
plt.plot(range(len(smoothed_rewards)), smoothed_rewards)

### Save the model using pytorch

In [ ]:
torch.save(model, 'models/dqn_breakout_r9.pt')